# 第5章 高次元データの数理 ― デモノートブック

講義ノート第5章の主張——次元の呪い、測度の集中、Johnson--Lindenstrauss の補題、
Marchenko--Pastur 則、BBP 相転移、共分散の縮小推定——を、すべて自分で走らせて確かめる。
とくに本章の核心である「$d/n\to\gamma>0$ を保つ限り、$n$ をいくら増やしても
消えない誤差が残る」（本文 5.5〜5.7 節）を、標本固有値と主成分方向の両方で見る。

## 目次

- [5.1 次元の呪い：体積はどこへ行ったか](#51)（本文 5.2 節、命題 5.1、図5.1）
- [5.2 ノルムと距離の集中](#52)（本文 5.3 節、定理 5.4、補題 5.6、図5.2）
- [5.3 Johnson--Lindenstrauss の補題](#53)（本文 5.4 節、定理 5.9、図5.3）
- [5.4 Marchenko--Pastur 則](#54)（本文 5.5 節、定理 5.12、図5.4）
- [5.5 BBP 相転移](#55)（本文 5.6 節、定理 5.15、図5.5）
- [5.6 縮小推定（Ledoit--Wolf）](#56)（本文 5.7 節、定義 5.17、定理 5.18、図5.6）
- [5.7 主成分方向の一致性はいつ回復するか](#57)（本文 5.6 節の含意）
- [演習](#ex) / [演習の解答](#sol)

データ行列は本講義の規約どおり**列がサンプル**（$\boldsymbol{X}\in\mathbb{R}^{d\times n}$）である。
`scipy` の `pdist` や `scikit-learn` は行がサンプルの規約なので、渡すときに転置する。
全セルを上から順に実行しておよそ 1〜2 分である。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

<a name="51"></a>
## 5.1 次元の呪い：体積はどこへ行ったか

本文 5.2 節。単位球の体積 $V_d=\pi^{d/2}/\Gamma(d/2+1)$（式 (5.1)）が $d\to\infty$ で
消えること（命題 5.1）と、体積が薄い外殻に集まること（式 (5.2)）を数値で見る。
図5.1 に対応する。例 5.2 の「近傍が空になる」計算も一緒に走らせる。
$\Gamma$ は大きな $d$ で溢れるので、対数（`gammaln`）を経由して計算する。

In [ ]:
from scipy.special import gammaln


def ball_volume(d):
    """単位球の体積 V_d = pi^{d/2}/Gamma(d/2+1)（式 (5.1)）。"""
    return np.exp(0.5 * d * np.log(np.pi) - gammaln(0.5 * d + 1.0))


print("  d        V_d       V_d / 2^d（立方体に対する割合）")
for d in [1, 2, 3, 5, 10, 20, 50]:
    print(f"{d:3d}  {ball_volume(d):11.4e}  {ball_volume(d) / 2.0**d:11.4e}")

dgrid = np.arange(1, 61)
V = ball_volume(dgrid)
d_star = int(dgrid[np.argmax(V)])
print(f"\nV_d が最大になる次元 d = {d_star}（V = {V.max():.4f}）")

print("\n外側 eps の殻が体積に占める割合 1-(1-eps)^d（式 (5.2)）")
for eps_s in [0.01, 0.05]:
    row = "   ".join(f"d={dd}: {1 - (1 - eps_s)**dd:.6f}" for dd in [10, 100, 1000])
    print(f"  eps={eps_s:.2f}   {row}")

n_pts, d_ex = 10**6, 100
s_side = (10 / n_pts) ** (1 / d_ex)
print(f"\n例 5.2：n={n_pts} 点、d={d_ex} で 10 点を含む小立方体の一辺 s = {s_side:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
axes[0].semilogy(dgrid, V, "o-", ms=3, color=C["blue"])
axes[0].axvline(d_star, ls=":", color=C["red"])
axes[0].set_xlabel(L("次元 d", "dimension d"))
axes[0].set_ylabel(L("単位球の体積 $V_d$", "volume $V_d$ of unit ball"))
axes[0].set_title(L("体積は $d=5$ を頂点に消えていく", "volume vanishes, peak at $d=5$"))
for eps_s, col in [(0.01, C["blue"]), (0.05, C["green"]), (0.10, C["orange"])]:
    axes[1].plot(dgrid, 1 - (1 - eps_s) ** dgrid, color=col, label=fr"$\epsilon={eps_s}$")
axes[1].set_xlabel(L("次元 d", "dimension d"))
axes[1].set_ylabel(L("外殻の占める割合", "fraction in outer shell"))
axes[1].set_title(L("体積は薄い殻に集まる", "volume concentrates in a thin shell"))
axes[1].legend()
fig.tight_layout()
plt.show()

$V_d$ は $d=5$ の 5.2638 を頂点に減少に転じ、$d=50$ では
1.730e-13 まで落ちる。立方体 $[-1,1]^d$ に対する比 $V_d/2^d$ は $d=20$ で
2.461e-08——**高次元立方体の体積はほとんどすべてが「隅」にある**ので、
立方体から一様に点を取ればほぼ確実に内接球の外側である。

外殻の割合 $1-(1-\epsilon)^d$ は $\epsilon=0.01$、$d=1000$ で 0.999957。
つまり単位球の体積の 99.99% 以上が厚さ 1% の殻に入る。高次元の球は
「詰まった玉」ではなく中身のない「殻」である。例 5.2 も同じ結論を指す：
$n=10^6$ 点を $d=100$ 次元に散らしたとき 10 点を含む小立方体の一辺は
$s=0.8913$、各辺の 89.1% が必要で、これは「局所近傍」ではなくほぼ定義域全体である。
$k$ 近傍法やカーネル密度推定が前提とする局所性はここで失われる。

<a name="52"></a>
## 5.2 ノルムと距離の集中

本文 5.3 節。$\boldsymbol{x}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}_d)$ のノルムが $\sqrt d$ のまわりに
$O(1)$ の幅で集中し（定理 5.4）、その帰結としてすべての点対距離がほぼ等しくなること
（命題 5.7）を見る。図5.2 に対応する。あわせて Laurent--Massart の相対版
（補題 5.6、式 (5.5)）
$\Pr(|Z/d-1|\ge\epsilon)\le2\exp\{-d(\epsilon^2-\epsilon^3)/4\}$ を実測と比べる。

In [ ]:
from scipy.spatial.distance import pdist

rng = np.random.default_rng(0)
dims = [2, 10, 100, 1000]
npts = 200
dist_scaled, ratio, sd_norm = {}, {}, {}
print("    d   ||x||/sqrt(d) の平均   標準偏差   最遠/最近の距離比")
for dd in dims:
    Xd = rng.standard_normal((dd, npts))     # d×n：列がサンプル
    nrm = np.linalg.norm(Xd, axis=0) / np.sqrt(dd)
    Dp = pdist(Xd.T)                         # pdist は n×d 規約なので転置して渡す
    dist_scaled[dd] = Dp / np.sqrt(2 * dd)
    ratio[dd] = float(Dp.max() / Dp.min())
    sd_norm[dd] = float(nrm.std())
    print(f"{dd:5d}          {nrm.mean():.4f}         {nrm.std():.4f}        {ratio[dd]:.4f}")

d_lm, n_mc = 1000, 20000
Zchi = rng.chisquare(d_lm, size=n_mc)        # Z ~ chi^2_d は ||x||^2 と同分布
print(f"\nLaurent--Massart の界（式 (5.5)、d={d_lm}、{n_mc} 回）と実測")
lm_rows = []
for e in [0.05, 0.10, 0.20, 0.30]:
    emp = float(np.mean(np.abs(Zchi / d_lm - 1) >= e))
    bnd = float(min(2 * np.exp(-d_lm * (e**2 - e**3) / 4), 1.0))
    lm_rows.append((e, emp, bnd))
    print(f"  eps={e:.2f}   実測 P={emp:.5f}   界 {bnd:.5e}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
for dd, col in zip(dims, [C["gray"], C["green"], C["orange"], C["blue"]]):
    axes[0].hist(dist_scaled[dd], bins=60, density=True, alpha=0.55, color=col,
                 label=f"d={dd}")
axes[0].axvline(1.0, ls=":", color=C["red"])
axes[0].set_xlabel(L(r"点対距離 / $\sqrt{2d}$", r"pairwise distance / $\sqrt{2d}$"))
axes[0].set_ylabel(L("密度", "density"))
axes[0].set_title(L("距離の集中（図5.2）", "concentration of distances (Fig. 5.2)"))
axes[0].legend()
axes[1].loglog(dims, [ratio[dd] for dd in dims], "o-", color=C["blue"])
axes[1].axhline(1.0, ls=":", color=C["red"])
axes[1].set_xlabel(L("次元 d", "dimension d"))
axes[1].set_ylabel(L("最遠 / 最近", "farthest / nearest"))
axes[1].set_title(L("最近傍と最遠点の比は 1 に近づく", "the ratio tends to 1"))
fig.tight_layout()
plt.show()

$\|\boldsymbol{x}\|/\sqrt d$ の標準偏差は $d=2$ の 0.4541 から
$d=1000$ の 0.0209 へ $O(d^{-1/2})$ で縮む——長さの**絶対的な**ばらつきが
$d$ に依らず $O(1)$ だからである（定理 5.4）。

その結果、$n=200$ 点での最遠 / 最近の距離比は $d=2$ で 1360.511、
$d=10$ で 7.144、$d=100$ で 1.874、$d=1000$ で 1.178 と 1 に近づく。
$d=1000$ では最も近い点と最も遠い点の距離が 17.8% しか違わない。
ノイズや丸め誤差で順位が容易に入れ替わるので、最近傍探索はここで意味を失う（命題 5.7）。

Laurent--Massart の界は実測をきちんと上から押さえる（$d=1000$）：
$\epsilon=0.10$ で実測 0.02540 に対し界 0.2108、
$\epsilon=0.30$ では実測 0.00000（20000 回中一度も起きない）に対し界
2.89e-07。界は保守的だが $\epsilon$ と $d$ について**指数的に**小さくなる。
この指数性が、次節で $n^2$ 個の事象に union bound を取るときに効く
（多項式界の Chebyshev では $n$ が数十で破綻する）。

<a name="53"></a>
## 5.3 Johnson--Lindenstrauss の補題

本文 5.4 節、定理 5.9。$n$ 点の対距離をすべて相対誤差 $\epsilon$ 以内で保つのに必要な
射影次元は $m=O(\epsilon^{-2}\log n)$ で、**元の次元 $d$ に全く依存しない**。
講義ノートのリスト（`lst:ch4-jl`）を走らせ、箱ひげ図（図5.3）を足す。
射影行列 $\boldsymbol{R}\in\mathbb{R}^{m\times d}$ は**左から**掛ける：
$(m\times d)(d\times n)=(m\times n)$ で、列がサンプルという規約が射影の前後で保たれ、
$\boldsymbol{R}$ が変数の側だけに作用することがひと目で分かる。

In [ ]:
rng = np.random.default_rng(1)
n_jl, d_jl, eps_jl = 500, 2000, 0.3
Xjl = rng.standard_normal((d_jl, n_jl))              # d×n：列がサンプル
dist0 = pdist(Xjl.T)                                 # pdist は n×d 規約なので転置
m_theory = int(np.ceil(24 * np.log(n_jl) / eps_jl**2))
print(f"n={n_jl}, d={d_jl}, eps={eps_jl}: 定理 5.9 が要求する m = {m_theory}")

ms = [50, 100, 200, 400, 800, m_theory]
distort, jl_max, jl_p95 = {}, {}, {}
for m in ms:
    R = rng.standard_normal((m, d_jl)) / np.sqrt(m)  # R は m×d、成分は N(0,1/m)
    dist1 = pdist((R @ Xjl).T)                       # 射影は左から：R X は m×n
    r = dist1 / dist0 - 1.0
    distort[m] = r
    jl_max[m] = float(np.abs(r).max())
    jl_p95[m] = float(np.percentile(np.abs(r), 95))
    print(f"  m={m:5d}   最大歪み={jl_max[m]:.4f}   95 パーセンタイル={jl_p95[m]:.4f}")

slope = float(np.polyfit(np.log(ms), np.log([jl_p95[m] for m in ms]), 1)[0])
print(f"\nlog(95 パーセンタイル歪み) 対 log(m) の傾き = {slope:.3f}（理論は -0.5）")

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.boxplot([distort[m] for m in ms], showfliers=False,
           medianprops=dict(color=C["red"]))
ax.set_xticks(range(1, len(ms) + 1))
ax.set_xticklabels([str(m) for m in ms])
ax.axhline(eps_jl, ls="--", color=C["red"])
ax.axhline(-eps_jl, ls="--", color=C["red"], label=fr"$\pm\epsilon={eps_jl}$")
ax.axhline(0.0, ls=":", color=C["gray"])
ax.set_xlabel(L("射影次元 m", "projection dimension m"))
ax.set_ylabel(L("相対歪み $d_1/d_0-1$", "relative distortion $d_1/d_0-1$"))
ax.set_title(L("Johnson--Lindenstrauss の数値検証（図5.3）",
               "Johnson-Lindenstrauss check (Fig. 5.3)"))
ax.legend()
fig.tight_layout()
plt.show()

$n=500$、$d=2000$、$\epsilon=0.3$ で定理 5.9 が要求する射影次元は $m=1658$。
最大歪みは $m=50$ で 0.4189、$m=100$ で 0.3025、$m=200$ で 0.2394、
$m=400$ で 0.1537、$m=800$ で 0.1085、$m=1658$ で 0.0816 である。
$m$ を 4 倍するとおよそ半分になり、歪み $\propto m^{-1/2}$ という理論の予言と整合する
（95 パーセンタイルに直線を当てると傾き -0.501、理論は $-0.5$）。

理論値 $m=1658$ は「全対の最悪ケースを確率 $1-1/n$ 以上で保証する」ための値なので保守的で、
$\epsilon=0.3$ の保証に対し実測の最大歪みは 0.0816 にすぎない。
95 パーセンタイルで見るなら $m=200$（理論値の約 1/8）でも歪みは 0.0989 に収まる。
なお JL の射影は**データを見ずに**決まる点で PCA と正反対である（本文 注意 5.11）。

<a name="54"></a>
## 5.4 Marchenko--Pastur 則

本文 5.5 節、定理 5.12。**真の共分散が単位行列**（相関構造がまったくない）でも、
$\gamma=d/n>0$ なら標本固有値は
$[\lambda_-,\lambda_+]=[(1-\sqrt\gamma)^2,\ (1+\sqrt\gamma)^2]$ に広がる。
密度は式 (5.9) である。図5.4 に対応する。ヒストグラムを滑らかにするため、
$d=400$ に固定して $n=d/\gamma$ とし、独立な実現を 5 回ぶん重ねて描く。

In [ ]:
trapz = getattr(np, "trapezoid", None) or np.trapz    # numpy 1.x / 2.x 両対応


def mp_density(x, gam):
    """Marchenko--Pastur 密度（式 (5.9)）。"""
    lm, lp = (1 - np.sqrt(gam))**2, (1 + np.sqrt(gam))**2
    xs = np.maximum(x, 1e-12)                      # gamma=1 では x=0 で発散する
    return np.where((x > lm) & (x < lp),
                    np.sqrt(np.maximum((lp - x) * (x - lm), 0.0)) / (2*np.pi*gam*xs), 0.0)


rng = np.random.default_rng(0)
d_mp, reps = 400, 5
mp_rows = []
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4))
for ax, gam in zip(axes, [0.1, 0.5, 1.0]):
    n_mp = int(round(d_mp / gam))
    ev = np.concatenate([
        np.linalg.eigvalsh(Xg @ Xg.T / n_mp)              # 標本共分散（d×d）
        for Xg in (rng.standard_normal((d_mp, n_mp))      # d×n：真の共分散は I_d
                   for _ in range(reps))])
    lm, lp = (1 - np.sqrt(gam))**2, (1 + np.sqrt(gam))**2
    grid = np.linspace(lm, lp, 4001)
    u = np.linspace(np.sqrt(lm), np.sqrt(lp), 4001)     # x=u^2 と置換して端点の
    integ = float(trapz(mp_density(u**2, gam) * 2 * u, u))   # 発散を回避して積分
    mp_rows.append((gam, d_mp, n_mp, lm, lp, float(ev.min()), float(ev.max()), integ))
    print(f"gamma={gam:.1f} (d={d_mp}, n={n_mp}, {reps} 回): 理論の台 [{lm:.4f}, {lp:.4f}]"
          f"  実測 [{ev.min():.4f}, {ev.max():.4f}]  密度の積分 {integ:.5f}")
    ax.hist(ev, bins=70, density=True, color=C["blue"], alpha=0.55)
    ax.plot(grid, mp_density(grid, gam), color=C["red"], lw=1.8)
    ax.set_title(fr"$\gamma={gam}$   $\lambda_\pm$=({lm:.3f}, {lp:.3f})", fontsize=10)
    ax.set_xlabel(L("固有値", "eigenvalue"))
axes[0].set_ylabel(L("密度", "density"))
axes[2].set_ylim(0, 1.2)
fig.suptitle(L("Marchenko--Pastur 則（図5.4）：真の共分散は $I_d$ なのに固有値は広がる",
               "Marchenko-Pastur law (Fig. 5.4): true covariance is $I_d$"))
fig.tight_layout()
plt.show()

$d=400$ 固定、5 回の実現をまとめた結果：

| $\gamma$ | $n$ | 理論の台 $[\lambda_-,\lambda_+]$ | 実測の範囲 | 密度の数値積分 |
|---|---|---|---|---|
| 0.1 | 4000 | $[0.4675,\ 1.7325]$ | $[0.4645,\ 1.7283]$ | 1.00000 |
| 0.5 | 800 | $[0.0858,\ 2.9142]$ | $[0.0832,\ 2.8834]$ | 0.99999 |
| 1.0 | 400 | $[0.0000,\ 4.0000]$ | $[0.0000,\ 3.9706]$ | 0.99984 |

理論の台と実測の範囲はよく一致する。$\gamma=0.5$ では真の固有値がすべて 1 なのに
標本固有値は 0.0832 から 2.8834 まで
**35 倍**散らばり、$\gamma=1$ では下端が 0 に達して
標本共分散はほぼ特異になる。密度の数値積分がいずれも 1 に近いことから、
式 (5.9) が確率密度であることも確かめられる。

**スクリープロットの読み方**：大きな固有値が見えてもそれが信号の証拠とは限らない。
$\gamma=0.5$ なら完全に構造のないノイズでも最大固有値は
$\lambda_+=2.914$ になる。判定の基準は「1 より大きいか」ではなく
「$\lambda_+=(1+\sqrt\gamma)^2$ より大きいか」である。

<a name="55"></a>
## 5.5 BBP 相転移

本文 5.6 節、定理 5.15（式 (5.12)）。スパイク共分散
$\boldsymbol{\Sigma}=\boldsymbol{I}_d+\theta\boldsymbol{v}\boldsymbol{v}^\top$（式 (5.11)）からデータを作り、
信号強度 $\theta$ を閾値 $\theta_{\rm c}=\sqrt\gamma$ の前後で動かす。
$\theta\le\sqrt\gamma$ では第一主成分が真の方向と**漸近的に直交**し、
$n$ を増やしても $\gamma$ を保つ限り改善しない。図5.5 に対応する。
階数 1 の摂動なので $\boldsymbol{\Sigma}^{1/2}=\boldsymbol{I}_d+(\sqrt{1+\theta}-1)\boldsymbol{v}\boldsymbol{v}^\top$ と
陽に書け、$d\times d$ 行列の平方根を作らずに
$\boldsymbol{X}=\boldsymbol{Z}+(\sqrt{1+\theta}-1)\boldsymbol{v}(\boldsymbol{v}^\top\boldsymbol{Z})$ で生成できる。

In [ ]:
def spike_experiment(theta, n=2000, d=1000, seed=0):
    """スパイク共分散 I_d + theta v v^T（式 (5.11)）からの最大固有値と overlap^2。"""
    rng = np.random.default_rng(seed)
    v = np.zeros(d); v[0] = 1.0
    Z = rng.standard_normal((d, n))                        # d×n：列がサンプル
    X = Z + (np.sqrt(1 + theta) - 1) * np.outer(v, v @ Z)  # X = Sigma^{1/2} Z
    S = X @ X.T / n                                        # 標本共分散（d×d）
    lam, U = np.linalg.eigh(S)
    return float(lam[-1]), float((U[:, -1] @ v)**2)


def bbp_theory(theta, gam):
    """定理 5.15（式 (5.12)）の理論値 (lambda_1, overlap^2)。"""
    if theta > np.sqrt(gam):
        return (1 + theta) * (1 + gam / theta), (1 - gam / theta**2) / (1 + gam / theta)
    return (1 + np.sqrt(gam))**2, 0.0


gam0 = 0.5                                # gamma = d/n = 1000/2000
print(f"gamma={gam0}, 閾値 sqrt(gamma) = {np.sqrt(gam0):.4f}（n=2000, d=1000）")
print(" theta   lam1(実測)  lam1(理論)   ov^2(実測)  ov^2(理論)")
bbp_tab = []
for theta in [0.3, 0.5, 0.707, 1.0, 1.5, 2.0]:
    lam1, ov2 = spike_experiment(theta)
    lt, ot = bbp_theory(theta, gam0)
    bbp_tab.append((theta, lam1, lt, ov2, ot))
    print(f"{theta:6.3f}   {lam1:8.3f}   {lt:8.3f}    {ov2:8.3f}   {ot:8.3f}")

thetas = np.linspace(0.1, 2.5, 13)
n_s, d_s, n_seed = 800, 400, 3
lam_emp = np.zeros(len(thetas)); ov_emp = np.zeros(len(thetas))
for i, th in enumerate(thetas):
    res = [spike_experiment(th, n=n_s, d=d_s, seed=100 + s) for s in range(n_seed)]
    lam_emp[i] = np.mean([r[0] for r in res])
    ov_emp[i] = np.mean([r[1] for r in res])
th_fine = np.linspace(0.05, 2.5, 400)
lam_th = np.array([bbp_theory(t, gam0)[0] for t in th_fine])
ov_th = np.array([bbp_theory(t, gam0)[1] for t in th_fine])

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
axes[0].plot(th_fine, lam_th, color=C["red"], label=L("理論", "theory"))
axes[0].plot(thetas, lam_emp, "o", color=C["blue"],
             label=L(f"実測（n={n_s}, d={d_s}）", f"empirical (n={n_s}, d={d_s})"))
axes[0].axvline(np.sqrt(gam0), ls=":", color=C["gray"])
axes[0].set_xlabel(r"$\theta$"); axes[0].set_ylabel(r"$\hat\lambda_1$")
axes[0].set_title(L("最大固有値", "largest eigenvalue")); axes[0].legend(fontsize=9)
axes[1].plot(th_fine, ov_th, color=C["red"])
axes[1].plot(thetas, ov_emp, "o", color=C["blue"])
axes[1].axvline(np.sqrt(gam0), ls=":", color=C["gray"])
axes[1].text(np.sqrt(gam0) + 0.06, 0.05, r"$\theta_c=\sqrt{\gamma}$", color=C["gray"])
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel(L(r"重なり $\langle\hat v,v\rangle^2$", r"overlap$^2$"))
axes[1].set_title(L("真の方向との重なり", "overlap with the true direction"))
fig.suptitle(L("BBP 相転移（図5.5）", "BBP phase transition (Fig. 5.5)"))
fig.tight_layout()
plt.show()

i_c = int(np.argmin(np.abs(thetas - 0.7)))
print(f"\n閾値近傍 theta={thetas[i_c]:.2f} の実測 overlap^2 = {ov_emp[i_c]:.4f}"
      f"（理論は 0。n={n_s} の有限サイズ効果である）")

$\gamma=0.5$（$n=2000$、$d=1000$）の閾値は $\sqrt{0.5}=0.7071$。実測（括弧内が理論値）：

| $\theta$ | $\hat\lambda_1$ | $\langle\hat{\boldsymbol{v}},\boldsymbol{v}\rangle^2$ |
|---|---|---|
| 0.300 | 2.855  (2.914) | 0.000  (0.000) |
| 0.500 | 2.858  (2.914) | 0.035  (0.000) |
| 0.707 | 2.883  (2.914) | 0.120  (0.000) |
| 1.000 | 2.989  (3.000) | 0.346  (0.333) |
| 1.500 | 3.330  (3.333) | 0.594  (0.583) |
| 2.000 | 3.752  (3.750) | 0.707  (0.700) |

閾値の十分下（$\theta=0.3$）では重なりが 0.0000 と実質的に直交し、
最大固有値は理論の $\lambda_+=2.914$ に張り付いている。
$\theta=1.5,\ 2.0$ では理論値と小数第 2 位まで合う。
閾値 $\theta_{\rm c}=0.707$ の近傍（表の $\theta=0.707$ で重なり 0.120）で
0 から離れて見えるのは有限 $n$ の効果である。図の $n=800$、$d=400$ の実測でも
閾値近傍 $\theta=0.70$ の重なりは 0.1029 と 0 から浮いており、
遷移が鋭くなるのは $n\to\infty$ の極限だけである。

実務的な含意：$n=200$ サンプル、$d=20000$ 遺伝子なら $\gamma=100$ で閾値は
$\theta\approx10$、つまり第一主成分方向の分散が背景の 11 倍以上ないと検出できない。
変数を 2000 に絞れば $\gamma=10$、$\theta_{\rm c}\approx3.2$ に下がる。
**高次元では情報を捨てることが検出力を上げる**場面がある。

<a name="56"></a>
## 5.6 縮小推定（Ledoit--Wolf）

本文 5.7 節。MP 則が示した「大きい固有値は上に、小さい固有値は下に偏る」歪みを、
$\hat{\boldsymbol{\Sigma}}_{\rm LW}(\rho)=(1-\rho)\hat{\boldsymbol{\Sigma}}+\rho\,\mu\,\boldsymbol{I}_d$
（定義 5.17）で打ち消す。決定論的な target に対するオラクル縮小率は式 (5.14) の
$\rho^\star=\beta^2/(\alpha^2+\beta^2)$ で、実務ではその一致推定量
$\hat\rho=\hat\beta^2/\hat\delta^2$ を使う。例 5.19（$n=60$、$d=40$）を再現し、
真の共分散への Frobenius 距離が縮小で改善することと、損失 $L(\rho)$ の形を見る。
図5.6 に対応する。`sklearn.covariance.LedoitWolf` は $n\times d$ 規約なので
`fit(X.T)` と転置して渡す。

In [ ]:
from sklearn.covariance import LedoitWolf

rng = np.random.default_rng(0)
n_lw, d_lw2 = 60, 40                                 # gamma = 0.667
Xlw = rng.standard_normal((d_lw2, n_lw))             # d×n：真の共分散は I_40
H = np.eye(n_lw) - np.ones((n_lw, n_lw)) / n_lw
Xc = Xlw @ H                                         # 中心化は右から H
Shat = Xc @ Xc.T / n_lw                              # 標本共分散（d×d）


def lw_rho(Xc_, S_):
    """Ledoit--Wolf の縮小率の推定量 rho_hat = beta^2/delta^2。"""
    d_, n_ = Xc_.shape
    mu_ = np.trace(S_) / d_
    delta2_ = np.sum((S_ - mu_ * np.eye(d_))**2)
    beta2_ = min(np.mean([np.sum((np.outer(Xc_[:, i], Xc_[:, i]) - S_)**2)
                          for i in range(n_)]) / n_, delta2_)
    return beta2_ / delta2_, mu_


rho_hat, mu_lw = lw_rho(Xc, Shat)
S_shrunk = (1 - rho_hat) * Shat + rho_hat * mu_lw * np.eye(d_lw2)
rho_sk = float(LedoitWolf().fit(Xlw.T).shrinkage_)   # sklearn は n×d 規約なので転置

ev_s = np.linalg.eigvalsh(Shat)
ev_r = np.linalg.eigvalsh(S_shrunk)
print(f"縮小率 rho（自前）= {rho_hat:.4f}   sklearn = {rho_sk:.4f}"
      f"   差 = {abs(rho_hat - rho_sk):.2e}")
print(f"標本共分散の固有値 [{ev_s.min():.4f}, {ev_s.max():.4f}]"
      f"   条件数 {ev_s.max()/ev_s.min():.1f}")
print(f"縮小後の固有値     [{ev_r.min():.4f}, {ev_r.max():.4f}]"
      f"   条件数 {ev_r.max()/ev_r.min():.2f}")
err_s = float(np.linalg.norm(Shat - np.eye(d_lw2), "fro"))
err_r = float(np.linalg.norm(S_shrunk - np.eye(d_lw2), "fro"))
print(f"真の共分散 I_40 との Frobenius 距離： 標本 {err_s:.4f} → 縮小後 {err_r:.4f}"
      f"（{100*(1-err_r/err_s):.1f}% 改善）")

# 真の共分散が単位行列だと target mu*I がほぼ真値なので rho=1 が最良になってしまう。
# 縮小率に内点の最適値が現れる設定として、固有値が 0.2〜5 に散った Sigma を使う。
lam_true = np.geomspace(0.2, 5.0, d_lw2)
Sig_true = np.diag(lam_true)
rhos = np.linspace(0, 1, 51)
n_trial = 200
loss = np.zeros_like(rhos)
rho_hats = []
for t in range(n_trial):
    Zt = np.random.default_rng(1000 + t).standard_normal((d_lw2, n_lw))
    Xt = (np.sqrt(lam_true)[:, None] * Zt) @ H     # X = Sigma^{1/2} Z を中心化（d×n）
    St = Xt @ Xt.T / n_lw
    mt = np.trace(St) / d_lw2
    for j, r in enumerate(rhos):
        loss[j] += np.sum(((1 - r) * St + r * mt * np.eye(d_lw2) - Sig_true)**2)
    rho_hats.append(lw_rho(Xt, St)[0])
loss /= n_trial
rho_best = float(rhos[int(np.argmin(loss))])
rho_hat_mean = float(np.mean(rho_hats))
print(f"\n真の固有値が 0.2〜5 に散った Sigma での {n_trial} 回平均の損失")
print(f"  損失 E||Sigma_hat(rho)-Sigma||_F^2 を最小にする rho = {rho_best:.2f}"
      f"（損失 {loss.min():.2f}、rho=0 では {loss[0]:.2f}、rho=1 では {loss[-1]:.2f}）")
print(f"  データから推定した rho_hat の平均 = {rho_hat_mean:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
idx = np.arange(1, d_lw2 + 1)
axes[0].plot(idx, ev_s[::-1], "o-", ms=4, color=C["blue"], label=L("標本共分散", "sample"))
axes[0].plot(idx, ev_r[::-1], "s-", ms=4, color=C["green"], label=L("縮小後", "shrunk"))
axes[0].axhline(1.0, ls="--", color=C["red"], label=L("真値", "truth"))
axes[0].set_xlabel(L("順位", "index")); axes[0].set_ylabel(L("固有値", "eigenvalue"))
axes[0].set_title(L("縮小は固有値の開きを畳む（図5.6）",
                    "shrinkage closes the spread (Fig. 5.6)"))
axes[0].legend(fontsize=9)
axes[1].plot(rhos, loss, color=C["blue"])
axes[1].axvline(rho_best, ls="--", color=C["red"],
                label=L(f"損失最小 {rho_best:.2f}", f"best {rho_best:.2f}"))
axes[1].axvline(rho_hat_mean, ls=":", color=C["green"],
                label=L(f"推定 {rho_hat_mean:.2f}", f"estimated {rho_hat_mean:.2f}"))
axes[1].set_xlabel(r"$\rho$")
axes[1].set_ylabel(L(r"損失 $E\|\hat\Sigma(\rho)-\Sigma\|_F^2$",
                     r"loss $E\|\hat\Sigma(\rho)-\Sigma\|_F^2$"))
axes[1].set_title(L(r"損失は $\rho$ の凸関数（真値は $\Sigma\neq I$）",
                    r"loss is convex in $\rho$ (true $\Sigma\neq I$)"), fontsize=11)
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

推定された縮小率は $\hat\rho=0.9637$ で、`sklearn` の `shrinkage_`
（0.9637）と 2.2e-16 の精度で一致する。
標本共分散の固有値は $[0.0195,\ 2.9190]$（条件数 149.8）と
大きく開くのに対し、縮小後は $[0.9498,\ 1.0551]$
（条件数 1.11）と真値 1 の近くに集まる。真の共分散
$\boldsymbol{I}_{40}$ との Frobenius 距離は 5.1753 から 0.2109 へ
95.9% 改善した。縮小は大きい固有値を下げ小さい固有値を上げる向きに働き、
MP 則が予言する歪みをちょうど打ち消している。

ただし真の共分散が $\boldsymbol{I}$ そのものだと target $\mu\boldsymbol{I}$ がほぼ真値なので
$\rho=1$ が最良になってしまい、縮小率の選択が問題にならない。そこで右の図では
真の固有値が 0.2〜5 に散った $\boldsymbol{\Sigma}$ で損失 $L(\rho)$ を 200 回平均した。
定理 5.18 のとおり $L$ は $\rho$ の凸二次関数になり、最小は内点
$\rho=0.48$（損失 34.23）にある。両端は
$\rho=0$（標本共分散）で 63.91、$\rho=1$（等分散）で 71.82 だから、
縮小は損失を 46% 下げている。
データから推定した $\hat\rho$ の平均は 0.4615 で、
オラクルの最良値 0.48 をよく追えている。

注意：本文の警告のとおり $\mu=\frac{1}{d}\operatorname{tr}\hat{\boldsymbol{\Sigma}}$ は**データの関数**なので、
定理 5.18 の最適性をそのまま $\boldsymbol{T}=\mu\boldsymbol{I}$ に適用してはいけない。
定理 5.18 は $d$ が大きいときの**目標**を与える定理で、上の $\hat\rho$ の正当化は漸近的である。

<a name="57"></a>
## 5.7 主成分方向の一致性はいつ回復するか

BBP 相転移は「$\gamma$ を保つ限り $n$ を増やしても駄目」と言う。では何を変えれば
主成分方向は正しくなるのか。$\theta=1$ を固定したまま、

1. $d=200$ を固定して $n$ を増やす（$\gamma=d/n\to0$、古典的漸近）
2. $\gamma=0.5$ を保ったまま $n$ を増やす（比例的高次元漸近）

の二通りを比べる。理論値は定理 5.15 の重なりの式に $\gamma=d/n$ を入れたものである。

In [ ]:
theta0, d_fix = 1.5, 200
ns = [100, 200, 400, 800, 1600, 3200, 6400, 12800]
ov_fixd = [float(np.mean([spike_experiment(theta0, n=nn, d=d_fix, seed=200 + s)[1]
                          for s in range(3)])) for nn in ns]
ov_fixd_th = [bbp_theory(theta0, d_fix / nn)[1] for nn in ns]

ns2 = [200, 400, 800, 1600]
ov_fixg = [float(np.mean([spike_experiment(theta0, n=nn, d=nn // 2, seed=300 + s)[1]
                          for s in range(5)])) for nn in ns2]
ov_fixg_th = bbp_theory(theta0, 0.5)[1]

print(f"theta={theta0} 固定。(1) d={d_fix} 固定で n を増やす（gamma=d/n が減る）")
print("     n    gamma   overlap^2(実測)  overlap^2(理論)")
for nn, oe, ot in zip(ns, ov_fixd, ov_fixd_th):
    print(f"{nn:6d}  {d_fix/nn:7.4f}      {oe:.4f}           {ot:.4f}")
print(f"\n(2) gamma=0.5 を保ったまま n を増やす（理論値 {ov_fixg_th:.4f} で一定）")
print("     n      d   overlap^2(実測)")
for nn, oe in zip(ns2, ov_fixg):
    print(f"{nn:6d}  {nn//2:5d}      {oe:.4f}")

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.semilogx(ns, ov_fixd, "o-", color=C["blue"],
            label=L(f"d={d_fix} 固定（実測）", f"d={d_fix} fixed (empirical)"))
ax.semilogx(ns, ov_fixd_th, "--", color=C["red"], label=L("理論", "theory"))
ax.semilogx(ns2, ov_fixg, "s-", color=C["green"],
            label=L(r"$\gamma=0.5$ 固定（実測）", r"$\gamma=0.5$ fixed (empirical)"))
ax.axhline(ov_fixg_th, ls=":", color=C["green"])
ax.axhline(1.0, ls=":", color=C["gray"])
ax.set_ylim(0, 1.05)
ax.set_xlabel("n")
ax.set_ylabel(L(r"重なり $\langle\hat v,v\rangle^2$", r"overlap$^2$"))
ax.set_title(L(r"一致性が回復するのは $\gamma\to0$ のときだけ",
               r"consistency recovers only as $\gamma\to0$"))
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

(1) $d=200$ を固定して $n$ を増やすと、重なりは $n=100$（$\gamma=2.00$）の
0.2578 から $n=12800$（$\gamma=0.0156$）の 0.9810 まで
単調に 1 へ向かい、各点で理論値とよく合う。これが古典的漸近 $\gamma\to0$ での一致性である。

(2) 一方 $\gamma=0.5$ を保つと、$n$ を 200 から 1600 まで 8 倍にしても重なりは
0.5227 → 0.5430 と理論値 0.5833 のまわりで動かない。
**サンプルを増やしても、それに比例して変数を増やしているなら主成分方向は正しくならない。**
これが比例的高次元漸近の核心であり、実務では「$n$ を増やす」より
「$d$ を減らす」ほうが効く場面がある、と読める（5.5 節の含意と同じ）。

<a name="ex"></a>
## 演習

1. **($\star$) 単位球の体積と殻。** $\epsilon=0.05$ のとき、外殻の占める割合
   $1-(1-\epsilon)^d$ が 0.99 を超える最小の $d$ を求めよ。また $V_d$ を最大にする
   整数 $d$ を求めよ（本文 演習 5.1）。偶数列と奇数列を別に見ると理由が分かる。
2. **($\star\star$) Rademacher 射影。** JL の射影行列の成分を $\pm1/\sqrt m$ の一様な符号に
   変えても歪みがほとんど変わらないことを $m=200$ で確かめよ（本文 演習 5.7）。
   中心極限定理により、$d$ が大きければ各行の内積はガウスの場合とほぼ同じ挙動をする。
3. **($\star\star\star$) 閾値の移動。** $\gamma=0.1$ と $\gamma=2.0$ について
   $\theta=\tfrac12\sqrt\gamma,\ \sqrt\gamma,\ 2\sqrt\gamma$ での重なりを測り、
   閾値が $\sqrt\gamma$ とともに動くことを確かめよ。$\gamma>1$ では標本共分散に
   0 固有値がいくつ現れるかも数えよ（本文 演習 5.8）。

In [ ]:
# 演習 1：eps=0.05 で外殻の割合が 0.99 を超える最小の d と、V_d を最大にする d
# TODO: 1-(1-eps)**d > 0.99 を解く（対数を取る）。ball_volume も使う
# d_min = ...
# print(d_min, 1 - 0.95**d_min)

# 演習 2：Rademacher 射影（成分が ±1/sqrt(m) の一様な符号）
# TODO: R_r = rng2.choice([-1.0, 1.0], size=(m_e, d_jl)) / np.sqrt(m_e) を作り、
#       m_e=200 での 95 パーセンタイル歪みをガウス射影と比べる
#       （dist0 と Xjl は 5.3 節のものをそのまま使ってよい）

# 演習 3：gamma を変えたときの BBP 閾値の移動
# TODO: gamma in [0.1, 2.0] について、theta = 0.5*sqrt(gamma), sqrt(gamma),
#       2*sqrt(gamma) で spike_experiment を走らせ overlap^2 を比べる。
#       gamma=2.0 では標本共分散の 0 固有値の個数も数える

<a name="sol"></a>
## 演習の解答

In [ ]:
# --- 演習 1 ---
eps_e = 0.05
d_min = int(np.ceil(np.log(0.01) / np.log(1 - eps_e)))
V_all = ball_volume(np.arange(1, 31))
print(f"演習1: 外殻の割合が 0.99 を超える最小の d = {d_min}"
      f"（1-(0.95)^{d_min} = {1-(1-eps_e)**d_min:.6f}、"
      f"d={d_min-1} では {1-(1-eps_e)**(d_min-1):.6f}）")
print(f"       V_d を最大にする d = {int(np.argmax(V_all)) + 1}"
      f"（V_5 = {V_all.max():.4f}、V_6 = {ball_volume(6):.4f}）")

# --- 演習 2 ---
rng2 = np.random.default_rng(7)
m_e = 200
R_g = rng2.standard_normal((m_e, d_jl)) / np.sqrt(m_e)
R_r = rng2.choice([-1.0, 1.0], size=(m_e, d_jl)) / np.sqrt(m_e)
p95_g = float(np.percentile(np.abs(pdist((R_g @ Xjl).T) / dist0 - 1), 95))
p95_r = float(np.percentile(np.abs(pdist((R_r @ Xjl).T) / dist0 - 1), 95))
print(f"\n演習2: m={m_e} の 95 パーセンタイル歪み  ガウス {p95_g:.4f} / "
      f"Rademacher {p95_r:.4f}（差 {abs(p95_g-p95_r):.4f}）")

# --- 演習 3 ---
print("\n演習3: 閾値 sqrt(gamma) の移動（n=600、3 回平均）")
n_e = 600
ex3 = {}
for gam_e in [0.1, 2.0]:
    d_e = int(round(gam_e * n_e))
    th_c = np.sqrt(gam_e)
    rows = []
    for fac in [0.5, 1.0, 2.0]:
        th = fac * th_c
        ov = float(np.mean([spike_experiment(th, n=n_e, d=d_e, seed=400 + s)[1]
                            for s in range(3)]))
        rows.append((fac, th, ov, bbp_theory(th, gam_e)[1]))
    ex3[gam_e] = rows
    print(f"  gamma={gam_e} (d={d_e}, 閾値 {th_c:.4f})")
    for fac, th, ov, ot in rows:
        print(f"    theta={fac:.1f}*sqrt(gamma)={th:.3f}: ov^2 実測 {ov:.4f} / 理論 {ot:.4f}")
Ze = np.random.default_rng(0).standard_normal((1200, n_e))     # gamma=2 の場合
ev_e = np.linalg.eigvalsh(Ze @ Ze.T / n_e)
n_zero = int(np.sum(ev_e < 1e-10))
print(f"  gamma=2.0 では d>n なので標本共分散は特異：0 固有値が {n_zero} 個"
      f"（d-n = {1200 - n_e}）")

**演習 1**：$0.95^d<0.01$ より $d>\log0.01/\log0.95=89.78$、
よって最小は $d=90$（実際 $1-0.95^{90}=0.990112$、
$d=89$ では 0.989591）。$V_d$ の最大は $d=5$ で
$V_5=5.2638$、$V_6=5.1677$ と僅差である
（比 $V_d/V_{d-2}=2\pi/d$ より偶数列の最大は $d=6$、奇数列は $d=5$）。

**演習 2**：$m=200$ の 95 パーセンタイル歪みはガウス射影 0.1012、
Rademacher 0.0970 で、差は 0.0042——小数第 3 位でしか違わない。
$\{-1,0,+1\}$ を確率 $\tfrac16,\tfrac23,\tfrac16$ で取る Achlioptas の構成なら
非零要素が 1/3 になって行列積が約 3 倍速く、保証の次数も変わらない。

**演習 3**：$\gamma=0.1$（閾値 0.3162）では $\theta=\sqrt\gamma$ で
すでに重なり 0.129、$2\sqrt\gamma$ で 0.721 と立ち上がる。
$\gamma=2.0$（閾値 1.4142）では同じ倍率でも $\theta=\sqrt\gamma$ で 0.079、
$2\sqrt\gamma$ で 0.459 と鈍く、閾値の半分（$\theta=0.707$）では
0.0086 とほぼ直交である。閾値が $\sqrt\gamma$ とともに上がるのが確認できる
（閾値以下の実測値が 0 でないのは $n=600$ の有限サイズ効果）。
$\gamma=2$ では $d>n$ なので標本共分散は必ず特異で、0 固有値がちょうど
$d-n=600$ 個（実測 600 個）現れる。
$d$ を $n$ より大きくすると検出閾値が上がるので、事前に変数を絞ることが検出力を上げる。